In [ ]:
import jax
import os

In [ ]:
import sys
sys.path.append("../")

%load_ext autoreload
%autoreload 2

import pulse_simulator as ps

In [ ]:
from datetime import datetime
import functools
import itertools
import logging
import numpy as np
import pickle as pkl
import qiskit as qk
from qiskit import pulse
import qiskit_aer
import qiskit.providers.fake_provider as qk_fp
import qiskit_dynamics as qk_d
import qutip as qt
import re 
from scipy.integrate import simpson
from scipy.linalg import sqrtm, expm
from tqdm import tqdm
import warnings

import matplotlib.pyplot as plt

In [ ]:
Units = 1e9

# Undo units
GHz = 1 / Units
MHz = 1e3 / Units
μs = Units / 1e3
ns = Units

backend = qk_fp.FakeManila()
dt = backend.configuration().dt * ns
desired_1qb_duration = 225  # dt
desired_2qb_duration = 225 * 4  # dt

# Gate time in SI units (granularity restriction)
duration_1qb = ps.qiskit_closest_multiple_of_16(desired_1qb_duration * dt / ns, backend)
duration_2qb = ps.qiskit_closest_multiple_of_16(desired_2qb_duration * dt / ns, backend)

# Calibrate

In [ ]:
# Load shapes
fname = "../pico-pulses/saved-pulses-2024-02-16/a_single_qubit_gateset_R1e-5.csv"
pico_a = {k: v for k, v in enumerate(ps.from_csv(fname))}
pico_a[len(pico_a)] = np.ones(pico_a[0].shape)

names_1qb = ["X1", "X2", "SX1", "SX2", "I1", "I2"]
angles_1qb = [1.5 * np.pi, 0.5 * np.pi, 1.25 * np.pi, 0.25 * np.pi, 1.0, 0.0]

In [ ]:
cal_1qb_robust = ps.calibrate_gateset(
    pico_a,
    angles_1qb,
    duration_1qb,
    dt,
    samples=32,
    names=names_1qb
)

In [ ]:
cal_1qb_default = ps.default_single_qubit_gateset(
    names_1qb,
    duration_1qb,
    dt,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

ax = axes[0]
for i, (k, v) in enumerate(cal_1qb_robust.items()):
    ax.plot(ps.get_samples(v).real, label=names_1qb[i])
ax.legend(loc="upper right")

ax = axes[1]
for i, (k, v) in enumerate(cal_1qb_default.items()):
    ax.plot(ps.get_samples(v).real, label=names_1qb[i])
ax.legend(loc="upper right")

In [ ]:
# Load shapes
def fname(k):
    return f"../pico-pulses/saved-pulses-2024-01-16/a_two_qubit_{k}.csv"

base_2qb = {k: ps.from_csv(fname(k))[0] for k in ["default", "robust"]}
robust_2qb = {0: base_2qb["robust"], 1: base_2qb["robust"]}
default_2qb = {0: base_2qb["default"], 1: base_2qb["default"]}
names_2qb = ["+", "-"]

In [ ]:
cal_2qb_robust = ps.calibrate_gateset(
    robust_2qb,
    [0.125 * np.pi, -0.125 * np.pi],
    duration_2qb,
    dt,
    samples=32,
    names=names_2qb
)

In [ ]:
cal_2qb_default = ps.calibrate_gateset(
    default_2qb,
    [0.125 * np.pi, -0.125 * np.pi],
    duration_2qb,
    dt,
    samples=32,
    names=names_2qb
)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3))
for i, (k, v) in enumerate(cal_2qb_robust.items()):
    ax.plot(ps.get_samples(v).real, label=names_2qb[i])
ax.legend(loc="upper right")